In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
from sklearn.utils import resample

### Lecture des fichier csv

In [ ]:
BASE = "datalake/transfermarkt/"

In [ ]:
player_profiles = pd.read_csv(BASE + "player_profiles/player_profiles.csv")
player_market_values = pd.read_csv(BASE + "player_market_value/player_market_value.csv")
player_latest_market_values = pd.read_csv(BASE + "player_latest_market_value/player_latest_market_value.csv")
player_performances = pd.read_csv(BASE + "player_performances/player_performances.csv")


## Traitement des fichier
### Traitement sur les profils de joueurs

In [ ]:
player_profiles=player_profiles[player_profiles['date_of_death'].isna()]
player_profiles=player_profiles[player_profiles['current_club_name']!='Retired']
player_profiles["second_nationality"] = (player_profiles["citizenship"]
                                         .str.split("  ")   # séparation sur double espace
                                         .str[1])
player_profiles["first_nationality"] = (player_profiles["citizenship"]
                                         .str.split("  ")   # séparation sur double espace
                                         .str[0])
player_profiles = player_profiles.drop(['player_image_url','citizenship','date_of_death','name_in_home_country','social_media_url','place_of_birth','country_of_birth','height','is_eu','outfitter','player_agent_id','player_agent_name','contract_option','second_club_url','third_club_url','fourth_club_url'], axis=1)
player_profiles=player_profiles[['player_id', 'player_slug', 'player_name', 'date_of_birth',  'first_nationality','second_nationality','position',
       'main_position', 'foot', 'current_club_id', 'current_club_name',
       'joined', 'contract_expires', 'date_of_last_contract_extension',
       'on_loan_from_club_id', 'on_loan_from_club_name',
       'contract_there_expires', 'second_club_name', 'third_club_name',
       'fourth_club_name']]

### Traitement sur les performances de joueurs

In [ ]:
player_performances=player_performances[player_performances['season_name'].str.contains('/')]
player_performances=player_performances.groupby(['season_name','player_id','team_id','team_name']).agg({'nb_in_group':'sum','goals':'sum','assists':'sum','yellow_cards':'sum','direct_red_cards':'sum','goals_conceded':'sum','clean_sheets':'sum'}).reset_index()
player_performances = player_performances.sort_values('goals',ascending=False).drop_duplicates(subset=["player_id", "season_name"], keep="first")
player_performances.sort_values('player_id',ascending=False)
def convert_season(s):
    first = int(s.split("/")[0])
    if first < 27:
        return 2000 + first
    else:
        return 1900 + first
player_performances["year"] = player_performances["season_name"].apply(convert_season)

### Traitement sur les valeurs de joueurs

In [ ]:
player_market_values["year"] = pd.to_datetime(player_market_values["date_unix"]).dt.year
player_market_values=player_market_values.sort_values("value", ascending=False).drop_duplicates(subset=["player_id", "year"],keep="first")

In [ ]:
print("player_profiles :", player_profiles.shape)
print("player_market_values :", player_market_values.shape)
print("player_performances :", player_performances.shape)

In [ ]:
Dataset=pd.merge(player_market_values,player_profiles,
    on=["player_id"],
    how="left")
Dataset=pd.merge(Dataset,player_performances,
    on=["player_id","year"],
    how="left")
Dataset['age']=(Dataset['year'] - pd.to_datetime(Dataset['date_of_birth']).dt.year)

In [ ]:
Dataset=Dataset.drop(['date_unix','second_nationality','player_name','current_club_id','on_loan_from_club_id','on_loan_from_club_name','contract_there_expires','second_club_name','third_club_name','fourth_club_name','team_id'],axis=1)

## Detection des biais

In [ ]:
import pandas as pd
import numpy as np

def compute_bias_stats(df):

    print("\n======================")
    print("📌 1. DONNÉES MANQUANTES")
    print("======================")
    print(df.isna().mean().sort_values(ascending=False).apply(lambda x: f"{x:.1%}"))

    # Helper pour détecter sur-représentation
    def dominant_group(series, threshold=0.50):
        s = series.value_counts(normalize=True)
        if s.iloc[0] > threshold:
            return f"⚠️ Sur-représentation : '{s.index[0]}' = {s.iloc[0]:.1%}"
        return "OK"

    print("\n======================")
    print("📌 2. NATIONALITÉS")
    print("======================")
    print(df['first_nationality'].value_counts())
    print(dominant_group(df['first_nationality']))

    print("\n======================")
    print("📌 3. POSITIONS PRINCIPALES")
    print("======================")
    print(df['main_position'].value_counts())
    print(dominant_group(df['main_position']))

    print("\n======================")
    print("📌 4. PIED DOMINANT")
    print("======================")
    print(df['foot'].value_counts())
    print(dominant_group(df['foot'], threshold=0.70))  # seuil 70% pour pied dominant

    print("\n======================")
    print("📌 5. CLUBS")
    print("======================")
    print(df['current_club_name'].value_counts().head(10))  # top 10
    print(dominant_group(df['current_club_name'], threshold=0.40))

    print("\n======================")
    print("📌 6. ÂGE")
    print("======================")
    print(df['age'].describe())
    print("Répartition :")
    print(pd.cut(df['age'], bins=[15,18,22,26,30,40]).value_counts())

    print("\n======================")
    print("📌 7. ANALYSE DES VALEURS (€)")
    print("======================")
    print(df.groupby('first_nationality')['value'].mean().sort_values(ascending=False).head(10))
    print(df.groupby('main_position')['value'].mean().sort_values(ascending=False))

    print("\n======================")
    print("📌 8. STATISTIQUES DE PERFORMANCE")
    print("======================")
    perf_cols = ['goals','assists','yellow_cards','direct_red_cards','clean_sheets']
    print(df[perf_cols].describe())

    print("\n--- Performances par POSTE ---")
    print(df.groupby('main_position')[perf_cols].mean())

    print("\n--- Performances par CLUB (Top 10) ---")
    print(df.groupby('current_club_name')[perf_cols].mean().sort_values('goals', ascending=False).head(10))

    print("\n======================")
    print("📌 9. BIAIS TEMPORELS (année & saison)")
    print("======================")
    print("\nAnnées :")
    print(df['year'].value_counts())
    print(dominant_group(df['year']))

    print("\nSaisons :")
    print(df['season_name'].value_counts())
    print(dominant_group(df['season_name'], threshold=0.40))

    print("\n🎯 Analyse terminée.")


compute_bias_stats(Dataset)


# Suppression des biais

In [ ]:
#Suppression des joueurs sans club, Inconnu,Medialiga
Dataset_Without_biais=Dataset[~(Dataset['current_club_name'].isin(['Without Club','Unknown','Medialiga','Career break','Free Agent']))]


In [ ]:
#Suppression des nationalités avec moins de 30 joueurs
nat_counts = Dataset_Without_biais["first_nationality"].value_counts()
valid_nats = nat_counts[nat_counts >= 30].index
Dataset_Without_biais = Dataset_Without_biais[Dataset_Without_biais["first_nationality"].isin(valid_nats)]

In [ ]:
#Suppression des joueurs hors tranche d'age 15-42
Dataset_Without_biais = Dataset_Without_biais[Dataset_Without_biais["age"].between(15, 42)]

In [ ]:
#Supprimer les clubs avec très peu de joueurs
club_counts = Dataset_Without_biais["current_club_name"].value_counts()
valid_clubs = club_counts[club_counts >= 20].index
Dataset_Without_biais = Dataset_Without_biais[Dataset_Without_biais["current_club_name"].isin(valid_clubs)]

In [ ]:


min_size = Dataset_Without_biais["main_position"].value_counts().min()

balanced = []
for pos, group in Dataset_Without_biais.groupby("main_position"):
    group_bal = resample(group, replace=False, n_samples=min_size, random_state=42)
    balanced.append(group_bal)

Dataset_Without_biais = pd.concat(balanced).reset_index(drop=True)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("dataset_debiased.csv")

# Variables sportives/objectives
features = [
    "age", "goals", "assists", "yellow_cards", 
    "direct_red_cards", "clean_sheets", "main_position",
    "foot", "current_club_name"
]

target = "value"

# Transformation catégorielle + one-hot (important)
categorical = ["main_position", "foot", "current_club_name", "first_nationality"]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown="ignore"), categorical)
    ],
    remainder='passthrough'
)

X = df[features + ["first_nationality"]]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modèle linéaire (interprétable)
model = LinearRegression()

# Pipeline
from sklearn.pipeline import Pipeline
pipe = Pipeline(steps=[
    ('prep', preprocess),
    ('model', model)
])

pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)
mae = mean_absolute_error(y_test, pred)

print("MAE :", mae)
